# ⚡ NanoGEMM: Sub-Microsecond Matrix Multiplication Benchmark

[![GitHub](https://img.shields.io/badge/GitHub-eminsk%2Fnanogemm-blue?logo=github)](https://github.com/eminsk/nanogemm)
[![PyPI](https://img.shields.io/pypi/v/nanogemm.svg)](https://pypi.org/project/nanogemm/)
[![Dev.to](https://img.shields.io/badge/Dev.to-Article-0a0a0a?logo=devdotto)](https://dev.to/eminsk/how-i-beat-numpy-matrix-multiplication-by-28x-with-a-100kb-c-microkernel-82k)

This interactive notebook benchmarks **NanoGEMM** (bare-metal AVX2 register-tiled GEMM) against **NumPy** directly on Google Colab CPU.

In [ ]:
# 1. Install NanoGEMM from PyPI
!pip install --upgrade nanogemm

In [ ]:
import nanogemm as ng
import numpy as np
import time

print('Active Hardware ISA:', ng.get_simd_isa())

In [ ]:
# 2. Correctness & Bitwise Exactness Verification
A = (np.random.randint(-8, 9, (32, 64)) * 0.25).astype(np.float32)
B = (np.random.randint(-8, 9, (64, 48)) * 0.5).astype(np.float32)

C_np = A @ B
C_ng = ng.matmul(A, B)

raw_diff = float(np.max(np.abs(C_ng - C_np)))
bit_xor = int(np.max(np.bitwise_xor(C_ng.view(np.uint32), C_np.view(np.uint32))))

assert raw_diff == 0.0 and bit_xor == 0, 'Verification failed!'
print(f'✅ 100% Bitwise Exactness Verified! (diff: {raw_diff} EXACT, 0-bit mismatch across {C_ng.size} elements)')

In [ ]:
# 3. Benchmark vs NumPy
dimensions = [16, 32, 64, 128]
print(f"{'Dim':>8} | {'NumPy (µs)':>12} | {'NanoGEMM (µs)':>14} | {'Speedup':>12}")
print('-' * 55)

for d in dimensions:
    A = np.random.randn(d, d).astype(np.float32)
    B = np.random.randn(d, d).astype(np.float32)
    iters = 10000 if d <= 64 else 2000
    
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = A @ B
    t_np = ((time.perf_counter() - t0) / iters) * 1e6
    
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = ng.matmul(A, B)
    t_ng = ((time.perf_counter() - t0) / iters) * 1e6
    
    speedup = t_np / t_ng
    speedup_str = f"{speedup:.2f}x FASTER 🚀" if speedup > 1.0 else f"{speedup:.2f}x"
    print(f"{d:>4}x{d:<3} | {t_np:>10.2f} µs | {t_ng:>12.2f} µs | {speedup_str:>14}")
